In [ ]:
from datasets import load_dataset, Video
from huggingface_hub import login
import pandas as pd
import numpy as np
import cv2
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
from tqdm import tqdm
import tempfile
import warnings
from moviepy import VideoFileClip
warnings.filterwarnings('ignore')


In [ ]:
ds = load_dataset("AI4A-lab/RecruitView")
train_ds = ds['train']
train_ds = train_ds.cast_column("video", Video(decode=False))
print(f"Total records: {len(train_ds)}")

In [35]:
from moviepy import VideoFileClip


In [ ]:
def get_video_path(video_field):
    """Dataset ke video field se ek usable file path nikalta hai
    (chahe wo string path ho, dict ho, ya raw bytes ho)."""
    if isinstance(video_field, str):
        return video_field
    elif isinstance(video_field, dict) and 'path' in video_field:
        return video_field['path']
    elif isinstance(video_field, (bytes, bytearray)):
        tmp = tempfile.NamedTemporaryFile(suffix='.mp4', delete=False)
        tmp.write(video_field)
        tmp.close()
        return tmp.name
    else:
        raise ValueError(f"Unknown video field format: {type(video_field)}")

In [ ]:
def estimate_eye_contact(landmarks, frame_w, frame_h):
    """Simple heuristic: iris/eye landmark deviation from frame center.
    Returns 1 if 'looking at camera' (centered), 0 otherwise.
    Threshold-based - refine later if needed."""
    try:
        left_iris = landmarks.landmark[468]  # requires refine_landmarks=True
        x, y = left_iris.x * frame_w, left_iris.y * frame_h
        center_x, center_y = frame_w / 2, frame_h / 2
        dist = np.sqrt((x - center_x)**2 + (y - center_y)**2)
        threshold = frame_w * 0.15  # tune this
        return 1 if dist < threshold else 0
    except (IndexError, AttributeError):
        return None

In [ ]:
def convert_frame_to_mp_image(frame):
    """OpenCV BGR frame ko MediaPipe Image object mein convert karo."""
    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    return mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)


def estimate_eye_contact_v2(face_landmarks, frame_w, frame_h):
    """Gaze-based heuristic using iris position relative to eye corners."""
    try:
        lm = face_landmarks  # list of NormalizedLandmark

        left_iris = lm[468]
        left_inner = lm[133]
        left_outer = lm[33]

        right_iris = lm[473]
        right_inner = lm[362]
        right_outer = lm[263]

        def gaze_ratio(iris, inner, outer):
            eye_width = outer.x - inner.x
            if eye_width == 0:
                return 0.5
            return (iris.x - inner.x) / eye_width

        left_ratio = gaze_ratio(left_iris, left_inner, left_outer)
        right_ratio = gaze_ratio(right_iris, right_inner, right_outer)
        avg_ratio = (left_ratio + right_ratio) / 2

        return 1 if 0.35 <= avg_ratio <= 0.65 else 0
    except (IndexError, AttributeError):
        return None


def get_emotion_proxy_from_blendshapes(blendshapes):
    """Blendshape categories ko emotion-proxy scores mein map karo."""
    scores = {b.category_name: b.score for b in blendshapes}
    return {
        'smile_score': (scores.get('mouthSmileLeft', 0) + scores.get('mouthSmileRight', 0)) / 2,
        'frown_score': (scores.get('browDownLeft', 0) + scores.get('browDownRight', 0)) / 2,
        'eye_openness': 1 - ((scores.get('eyeBlinkLeft', 0) + scores.get('eyeBlinkRight', 0)) / 2),
        'jaw_open': scores.get('jawOpen', 0),
        'brow_raise': (scores.get('browOuterUpLeft', 0) + scores.get('browOuterUpRight', 0)) / 2,
        'mouth_frown': (scores.get('mouthFrownLeft', 0) + scores.get('mouthFrownRight', 0)) / 2,
    }


def extract_face_features(video_path, sample_rate_sec=1):
    """Extract per-frame face/emotion-proxy features using MediaPipe Tasks API."""
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    if fps == 0 or np.isnan(fps):
        fps = 25
    frame_interval = max(1, int(fps * sample_rate_sec))

    frame_records = []
    frame_idx = 0

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        if frame_idx % frame_interval == 0:
            h, w = frame.shape[:2]
            mp_image = convert_frame_to_mp_image(frame)

            result = detector.detect(mp_image)

            face_detected = len(result.face_landmarks) > 0
            eye_contact = None
            emotion_proxy = None

            if face_detected:
                landmarks = result.face_landmarks[0]
                eye_contact = estimate_eye_contact_v2(landmarks, w, h)

                if result.face_blendshapes:
                    emotion_proxy = get_emotion_proxy_from_blendshapes(result.face_blendshapes[0])

            frame_records.append({
                'face_detected': face_detected,
                'eye_contact': eye_contact,
                'emotion_proxy': emotion_proxy
            })
        frame_idx += 1

    cap.release()
    return frame_records

In [ ]:
with open("face_landmarker.task", "rb") as f:
    model_data = f.read()

base_options = python.BaseOptions(model_asset_buffer=model_data)  # path nahi, buffer
options = vision.FaceLandmarkerOptions(
    base_options=base_options,
    output_face_blendshapes=True,
    output_facial_transformation_matrixes=False,
    num_faces=1,
    running_mode=vision.RunningMode.IMAGE
)
detector = vision.FaceLandmarker.create_from_options(options)
print("Detector ready")

Detector ready


In [38]:
# temporarily test on one video's frames, print raw ratios
sample = train_ds[0]
video_path = get_video_path(sample['video'])

cap = cv2.VideoCapture(video_path)
ret, frame = cap.read()
mp_image = convert_frame_to_mp_image(frame)
result = detector.detect(mp_image)

if result.face_landmarks:
    lm = result.face_landmarks[0]
    left_iris, left_inner, left_outer = lm[468], lm[133], lm[33]
    right_iris, right_inner, right_outer = lm[473], lm[362], lm[263]
    
    left_ratio = (left_iris.x - left_inner.x) / (left_outer.x - left_inner.x)
    right_ratio = (right_iris.x - right_inner.x) / (right_outer.x - right_inner.x)
    print(f"Left ratio: {left_ratio:.3f}, Right ratio: {right_ratio:.3f}")
cap.release()

Left ratio: 0.549, Right ratio: 0.512


In [40]:
def estimate_gaze_ratio(face_landmarks):
    """Return raw gaze ratio (continuous), not binary. 
    ~0.5 = centered/camera-facing, deviation = looking away."""
    try:
        lm = face_landmarks
        left_iris, left_inner, left_outer = lm[468], lm[133], lm[33]
        right_iris, right_inner, right_outer = lm[473], lm[362], lm[263]

        def ratio(iris, inner, outer):
            w = outer.x - inner.x
            return (iris.x - inner.x) / w if w != 0 else 0.5

        left_ratio = ratio(left_iris, left_inner, left_outer)
        right_ratio = ratio(right_iris, right_inner, right_outer)
        return (left_ratio + right_ratio) / 2
    except (IndexError, AttributeError):
        return None

In [43]:
def extract_face_features(video_path, sample_rate_sec=1):
    """Extract per-frame face/emotion-proxy features using MediaPipe Tasks API."""
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    if fps == 0 or np.isnan(fps):
        fps = 25
    frame_interval = max(1, int(fps * sample_rate_sec))

    frame_records = []
    frame_idx = 0

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        if frame_idx % frame_interval == 0:
            h, w = frame.shape[:2]
            mp_image = convert_frame_to_mp_image(frame)

            result = detector.detect(mp_image)

            face_detected = len(result.face_landmarks) > 0
            gaze_ratio = None
            emotion_proxy = None

            if face_detected:
                landmarks = result.face_landmarks[0]
                gaze_ratio = estimate_gaze_ratio(landmarks)

                if result.face_blendshapes:
                    emotion_proxy = get_emotion_proxy_from_blendshapes(result.face_blendshapes[0])

            frame_records.append({
                'face_detected': face_detected,
                'gaze_ratio': gaze_ratio,
                'emotion_proxy': emotion_proxy
            })
        frame_idx += 1

    cap.release()
    return frame_records


In [45]:
def estimate_gaze_ratio(face_landmarks):
    """Return raw gaze ratio (continuous), not binary.
    ~0.5 = centered/camera-facing, deviation = looking away."""
    try:
        lm = face_landmarks
        left_iris, left_inner, left_outer = lm[468], lm[133], lm[33]
        right_iris, right_inner, right_outer = lm[473], lm[362], lm[263]

        def ratio(iris, inner, outer):
            w = outer.x - inner.x
            return (iris.x - inner.x) / w if w != 0 else 0.5

        left_ratio = ratio(left_iris, left_inner, left_outer)
        right_ratio = ratio(right_iris, right_inner, right_outer)
        return (left_ratio + right_ratio) / 2
    except (IndexError, AttributeError):
        return None

In [53]:
def blendshapes_to_emotions(blendshapes):
    """MediaPipe blendshapes ko 7 standard emotion probabilities mein map karta hai."""
    scores = {b.category_name: b.score for b in blendshapes}
    g = lambda k: scores.get(k, 0.0)

    happy = (g('mouthSmileLeft') + g('mouthSmileRight')) / 2 * (1 - g('jawOpen') * 0.3)
    sad = ((g('mouthFrownLeft') + g('mouthFrownRight')) / 2 * 0.6 +
           g('browInnerUp') * 0.4 +
           (g('mouthLowerDownLeft') + g('mouthLowerDownRight')) / 2 * 0.3)
    angry = ((g('browDownLeft') + g('browDownRight')) / 2 * 0.5 +
             (g('noseSneerLeft') + g('noseSneerRight')) / 2 * 0.3 +
             (g('mouthPressLeft') + g('mouthPressRight')) / 2 * 0.2)
    surprise = ((g('browOuterUpLeft') + g('browOuterUpRight')) / 2 * 0.4 +
                (g('eyeWideLeft') + g('eyeWideRight')) / 2 * 0.3 +
                g('jawOpen') * 0.3)
    fear = (g('browInnerUp') * 0.3 +
            (g('eyeWideLeft') + g('eyeWideRight')) / 2 * 0.4 +
            (g('mouthStretchLeft') + g('mouthStretchRight')) / 2 * 0.3)
    disgust = ((g('noseSneerLeft') + g('noseSneerRight')) / 2 * 0.5 +
               (g('mouthUpperUpLeft') + g('mouthUpperUpRight')) / 2 * 0.3 +
               (g('browDownLeft') + g('browDownRight')) / 2 * 0.2)

    raw_scores = {'happy': happy, 'sad': sad, 'angry': angry,
                  'surprise': surprise, 'fear': fear, 'disgust': disgust}
    total_activation = sum(raw_scores.values())
    raw_scores['neutral'] = max(0.0, 1 - total_activation)

    total = sum(raw_scores.values())
    if total > 0:
        return {k: v / total for k, v in raw_scores.items()}
    return {k: (1.0 if k == 'neutral' else 0.0) for k in raw_scores}

In [54]:
def extract_face_features(video_path, sample_rate_sec=1):
    """Extract per-frame face/emotion features using MediaPipe Tasks API."""
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    if fps == 0 or np.isnan(fps):
        fps = 25
    frame_interval = max(1, int(fps * sample_rate_sec))

    frame_records = []
    frame_idx = 0

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        if frame_idx % frame_interval == 0:
            h, w = frame.shape[:2]
            mp_image = convert_frame_to_mp_image(frame)
            result = detector.detect(mp_image)

            face_detected = len(result.face_landmarks) > 0
            gaze_ratio = None
            emotion_proxy = None
            emotion_7class = None

            if face_detected:
                landmarks = result.face_landmarks[0]
                gaze_ratio = estimate_gaze_ratio(landmarks)

                if result.face_blendshapes:
                    emotion_proxy = get_emotion_proxy_from_blendshapes(result.face_blendshapes[0])
                    emotion_7class = blendshapes_to_emotions(result.face_blendshapes[0])

            frame_records.append({
                'face_detected': face_detected,
                'gaze_ratio': gaze_ratio,
                'emotion_proxy': emotion_proxy,
                'emotion_7class': emotion_7class
            })
        frame_idx += 1

    cap.release()
    return frame_records

In [55]:
EMOTION_PROXY_KEYS = ['smile_score', 'frown_score', 'eye_openness', 'jaw_open', 'brow_raise', 'mouth_frown']
EMOTION_7_LABELS = ['happy', 'sad', 'angry', 'surprise', 'fear', 'disgust', 'neutral']

def aggregate_video_features(frame_records, video_id):
    total_frames = len(frame_records)
    if total_frames == 0:
        return {'id': video_id, 'face_detected_ratio': 0.0}

    face_detected_count = sum(1 for r in frame_records if r['face_detected'])
    gaze_vals = [r['gaze_ratio'] for r in frame_records if r['gaze_ratio'] is not None]

    result = {
        'id': video_id,
        'face_detected_ratio': face_detected_count / total_frames,
        'gaze_ratio_mean': np.mean(gaze_vals) if gaze_vals else np.nan,
        'gaze_deviation_mean': np.mean([abs(v - 0.5) for v in gaze_vals]) if gaze_vals else np.nan,
        'gaze_stability_std': np.std(gaze_vals) if gaze_vals else np.nan,
    }

    for key in EMOTION_PROXY_KEYS:
        vals = [r['emotion_proxy'][key] for r in frame_records if r['emotion_proxy'] is not None]
        result[f'{key}_mean'] = np.mean(vals) if vals else np.nan
        result[f'{key}_std'] = np.std(vals) if vals else np.nan

    for label in EMOTION_7_LABELS:
        vals = [r['emotion_7class'][label] for r in frame_records if r.get('emotion_7class') is not None]
        result[f'emotion_{label}_mean'] = np.mean(vals) if vals else np.nan

    return result

In [56]:
sample = train_ds[0]
video_path = get_video_path(sample['video'])
frame_records = extract_face_features(video_path, sample_rate_sec=1)
agg = aggregate_video_features(frame_records, sample['id'])
print(agg)

{'id': '0001', 'face_detected_ratio': 1.0, 'gaze_ratio_mean': np.float64(0.5259281626346135), 'gaze_deviation_mean': np.float64(0.02821083486828747), 'gaze_stability_std': np.float64(0.016503204820425815), 'smile_score_mean': np.float64(0.45887441840287385), 'smile_score_std': np.float64(0.25219741502022963), 'frown_score_mean': np.float64(0.0013334394271541069), 'frown_score_std': np.float64(0.001154940649545297), 'eye_openness_mean': np.float64(0.7671516596720639), 'eye_openness_std': np.float64(0.17557922610784069), 'jaw_open_mean': np.float64(0.015858630677526395), 'jaw_open_std': np.float64(0.02211229048848542), 'brow_raise_mean': np.float64(0.39942656321959064), 'brow_raise_std': np.float64(0.1047675455437204), 'mouth_frown_mean': np.float64(0.0017325413910860086), 'mouth_frown_std': np.float64(0.004994546144234892), 'emotion_happy_mean': np.float64(0.3860009004103384), 'emotion_sad_mean': np.float64(0.15330118908608165), 'emotion_angry_mean': np.float64(0.007711355055406602), 'e

In [57]:
CHECKPOINT_EVERY = 100
results = []
failed_ids = []

for i, sample in enumerate(tqdm(train_ds, desc="Processing full dataset with emotions")):
    video_id = sample['id']
    try:
        video_path = get_video_path(sample['video'])
        frame_records = extract_face_features(video_path, sample_rate_sec=1)
        agg_features = aggregate_video_features(frame_records, video_id)
        results.append(agg_features)
    except Exception as e:
        print(f"Failed on id={video_id}: {e}")
        failed_ids.append(video_id)

    if (i + 1) % CHECKPOINT_EVERY == 0:
        pd.DataFrame(results).to_csv('face_features_checkpoint.csv', index=False)
        print(f"Checkpoint saved at {i+1} videos")

face_df = pd.DataFrame(results)
face_df.to_csv('face_features_final.csv', index=False)
print("Done:", face_df.shape)

Processing full dataset with emotions:   5%|▍         | 100/2011 [03:26<1:01:08,  1.92s/it]

Checkpoint saved at 100 videos


Processing full dataset with emotions:  10%|▉         | 200/2011 [06:18<51:15,  1.70s/it]  

Checkpoint saved at 200 videos


Processing full dataset with emotions:  15%|█▍        | 300/2011 [09:34<1:10:00,  2.46s/it]

...

Checkpoint saved at 1900 videos


Processing full dataset with emotions:  99%|█████████▉| 2000/2011 [1:18:08<00:25,  2.28s/it]

Checkpoint saved at 2000 videos


Processing full dataset with emotions: 100%|██████████| 2011/2011 [1:18:33<00:00,  2.34s/it]

Done: (2011, 24)


In [74]:
with open("face_landmarker.task", "rb") as f:
    model_data = f.read()

base_options_v2 = python.BaseOptions(model_asset_buffer=model_data)
options_v2 = vision.FaceLandmarkerOptions(
    base_options=base_options_v2,
    output_face_blendshapes=False,   # is pass mein emotion nahi chahiye, already ho chuka
    output_facial_transformation_matrixes=True,   # head pose ke liye zaroori
    num_faces=1,
    running_mode=vision.RunningMode.IMAGE
)
detector_v2 = vision.FaceLandmarker.create_from_options(options_v2)

In [62]:
def compute_face_geometry(landmarks, frame_w, frame_h):
    xs = [lm.x * frame_w for lm in landmarks]
    ys = [lm.y * frame_h for lm in landmarks]
    face_width = max(xs) - min(xs)
    face_height = max(ys) - min(ys)
    return {
        'face_width': face_width,
        'face_height': face_height,
        'face_area': face_width * face_height,
        'face_center_x': (max(xs) + min(xs)) / 2,
        'face_center_y': (max(ys) + min(ys)) / 2
    }


def compute_ear(landmarks):
    def dist(p1, p2):
        return np.sqrt((p1.x - p2.x)**2 + (p1.y - p2.y)**2)
    left_ear = (dist(landmarks[160], landmarks[144]) + dist(landmarks[158], landmarks[153])) / \
                (2 * dist(landmarks[33], landmarks[133]))
    right_ear = (dist(landmarks[385], landmarks[380]) + dist(landmarks[387], landmarks[373])) / \
                 (2 * dist(landmarks[362], landmarks[263]))
    return (left_ear + right_ear) / 2


def compute_head_pose(transformation_matrix):
    m = transformation_matrix
    sy = np.sqrt(m[0][0]**2 + m[1][0]**2)
    singular = sy < 1e-6
    if not singular:
        pitch = np.arctan2(m[2][1], m[2][2])
        yaw = np.arctan2(-m[2][0], sy)
        roll = np.arctan2(m[1][0], m[0][0])
    else:
        pitch = np.arctan2(-m[1][2], m[1][1])
        yaw = np.arctan2(-m[2][0], sy)
        roll = 0
    return {'pitch': np.degrees(pitch), 'yaw': np.degrees(yaw), 'roll': np.degrees(roll)}

In [64]:
def extract_extra_features(video_path, sample_rate_sec=1):
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    if fps == 0 or np.isnan(fps):
        fps = 25
    frame_interval = max(1, int(fps * sample_rate_sec))

    frame_records = []
    frame_idx = 0

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        if frame_idx % frame_interval == 0:
            h, w = frame.shape[:2]
            mp_image = convert_frame_to_mp_image(frame)
            result = detector_v2.detect(mp_image)

            face_geometry = None
            ear = None
            head_pose = None

            if len(result.face_landmarks) > 0:
                landmarks = result.face_landmarks[0]
                face_geometry = compute_face_geometry(landmarks, w, h)
                ear = compute_ear(landmarks)
                if result.facial_transformation_matrixes:
                    head_pose = compute_head_pose(result.facial_transformation_matrixes[0])

            frame_records.append({
                'face_geometry': face_geometry,
                'ear': ear,
                'head_pose': head_pose
            })
        frame_idx += 1

    cap.release()
    return frame_records

In [65]:
def aggregate_extra_features(frame_records, video_id):
    result = {'id': video_id}

    geo_keys = ['face_width', 'face_height', 'face_area', 'face_center_x', 'face_center_y']
    for key in geo_keys:
        vals = [r['face_geometry'][key] for r in frame_records if r['face_geometry'] is not None]
        result[f'{key}_mean'] = np.mean(vals) if vals else np.nan

    ear_vals = [r['ear'] for r in frame_records if r['ear'] is not None]
    result['EAR_mean'] = np.mean(ear_vals) if ear_vals else np.nan
    result['EAR_std'] = np.std(ear_vals) if ear_vals else np.nan

    centers = [(r['face_geometry']['face_center_x'], r['face_geometry']['face_center_y'])
               for r in frame_records if r['face_geometry'] is not None]
    if len(centers) > 1:
        movements = [np.sqrt((centers[i][0]-centers[i-1][0])**2 + (centers[i][1]-centers[i-1][1])**2)
                     for i in range(1, len(centers))]
        result['head_motion_std'] = np.std(movements)
    else:
        result['head_motion_std'] = np.nan

    for angle in ['pitch', 'yaw', 'roll']:
        vals = [r['head_pose'][angle] for r in frame_records if r['head_pose'] is not None]
        result[f'{angle}_mean'] = np.mean(vals) if vals else np.nan
        result[f'{angle}_std'] = np.std(vals) if vals else np.nan

    return result

In [66]:
sample = train_ds[0]
video_path = get_video_path(sample['video'])
frame_records = extract_extra_features(video_path, sample_rate_sec=1)
agg = aggregate_extra_features(frame_records, sample['id'])
print(agg)

{'id': '0001', 'face_width_mean': np.float64(213.91643437472257), 'face_height_mean': np.float64(246.0898086157712), 'face_area_mean': np.float64(52752.95815565931), 'face_center_x_mean': np.float64(344.27194378592753), 'face_center_y_mean': np.float64(278.8277102058584), 'EAR_mean': np.float64(0.29495586635295407), 'EAR_std': np.float64(0.09834532792227856), 'head_motion_std': np.float64(8.008589492678997), 'pitch_mean': np.float64(-12.977782466271094), 'pitch_std': np.float64(4.468693408583492), 'yaw_mean': np.float64(-2.0602380520742254), 'yaw_std': np.float64(3.195966184899133), 'roll_mean': np.float64(-0.775706321466145), 'roll_std': np.float64(2.904071001399732)}


In [68]:
CHECKPOINT_EVERY = 100
results = []
failed_ids = []

for i, sample in enumerate(tqdm(train_ds, desc="Extracting extra features")):
    video_id = sample['id']
    try:
        video_path = get_video_path(sample['video'])
        frame_records = extract_extra_features(video_path, sample_rate_sec=1)
        agg_features = aggregate_extra_features(frame_records, video_id)
        results.append(agg_features)
    except Exception as e:
        print(f"Failed on id={video_id}: {e}")
        failed_ids.append(video_id)

    if (i + 1) % CHECKPOINT_EVERY == 0:
        pd.DataFrame(results).to_csv('face_extra_features_checkpoint.csv', index=False)
        print(f"Checkpoint saved at {i+1} videos")

extra_df = pd.DataFrame(results)
extra_df.to_csv('face_extra_features.csv', index=False)
print("Done:", extra_df.shape)

Extracting extra features:   5%|▍         | 100/2011 [03:46<1:16:03,  2.39s/it]

Checkpoint saved at 100 videos


Extracting extra features:  10%|▉         | 200/2011 [06:53<56:57,  1.89s/it]  

Checkpoint saved at 200 videos


Extracting extra features:  15%|█▍        | 300/2011 [11:17<1:13:23,  2.57s/it] 

...

Checkpoint saved at 1900 videos


Extracting extra features:  99%|█████████▉| 2000/2011 [1:17:57<00:24,  2.20s/it]

Checkpoint saved at 2000 videos


Extracting extra features: 100%|██████████| 2011/2011 [1:18:21<00:00,  2.34s/it]

Done: (2011, 15)


In [69]:
main_df = pd.read_csv('face_features_final.csv')    
extra_df = pd.read_csv('face_extra_features.csv')     
merged_df = main_df.merge(extra_df, on='id', how='inner')
merged_df.to_csv('face_features_complete.csv', index=False)
print(merged_df.shape)   

(2011, 38)


In [70]:
df = pd.read_csv("face_features_complete.csv")

In [73]:
df.head(5)

,id,face_detected_ratio,gaze_ratio_mean,gaze_deviation_mean,gaze_stability_std,smile_score_mean,smile_score_std,frown_score_mean,frown_score_std,eye_openness_mean,...,face_center_y_mean,EAR_mean,EAR_std,head_motion_std,pitch_mean,pitch_std,yaw_mean,yaw_std,roll_mean,roll_std
0,1,1.0,0.525928,0.028211,0.016503,0.458874,0.252197,0.001333,0.001155,0.767152,...,278.827710,0.294956,0.098345,8.008589,-12.977782,4.468693,-2.060238,3.195966,-0.775706,2.904071
1,2,1.0,0.542930,0.046900,0.021770,0.075525,0.071071,0.000263,0.000241,0.861586,...,305.164212,0.372335,0.094387,6.179175,-6.442817,3.607256,1.076124,2.769734,-6.338320,3.046731
2,3,1.0,0.546025,0.046025,0.008399,0.043278,0.093646,0.006341,0.005869,0.894525,...,262.926211,0.385815,0.028359,15.923207,2.550276,2.050158,-6.124523,2.153081,2.190924,2.193048
3,4,1.0,0.541986,0.042423,0.014829,0.000419,0.001847,0.012285,0.066007,0.920164,...,226.942159,0.389095,0.047019,12.334878,-9.680763,5.410127,-1.970349,4.525574,5.035455,13.224997
4,5,1.0,0.546883,0.048153,0.017564,0.043369,0.065311,0.026583,0.036279,0.875688,...,192.718945,0.376027,0.065716,13.125939,-11.526389,4.618452,-7.228292,2.928528,6.761659,4.138178
